# DynGraphEval — Training

Set `model` and `dataset` in the Config cell, then **Run All**.

| `model` | Description |
|---|---|
| `tgn` | TGN (TPNet implementation — 2-layer GraphAttention, full neighbor history) |

## 0. Setup

In [ ]:
import sys, os, subprocess

# ── 1. Mount Drive (persistent storage: checkpoints) ──────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/DynGraphEval"
CODE_DIR  = "/content/DynGraphEval"
REPO_URL  = "https://github.com/importjose/DynGraphEval.git"
BRANCH    = "feature/init"

# ── 2. Clone latest code from GitHub (always fresh — /content is ephemeral) 
os.chdir("/content")  # move out before deleting
subprocess.run(f"rm -rf {CODE_DIR}", shell=True, check=True)
subprocess.run(
    f"git clone --quiet --branch {BRANCH} {REPO_URL} {CODE_DIR}",
    shell=True, check=True, cwd="/content"
)

os.chdir(CODE_DIR)
print(f"Working directory: {os.getcwd()}")

# ── 3. Symlink checkpoints dir from Drive ─────────────────────────────────
for rel in ["models/tgn/checkpoints"]:
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(CODE_DIR, rel)
    os.makedirs(src, exist_ok=True)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        os.symlink(src, dst)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Config

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
model   = 'tgn'        # tgn
dataset = 'tgbl-wiki'  # tgbl-wiki | tgbl-review | tgbl-coin | ...

# ── Optional overrides (leave as None to use script defaults) ─────────────────
epochs   = None   # default: 50
patience = None   # default: 5
seed     = None   # default: 0
device   = None   # default: auto (cuda if available)
# ─────────────────────────────────────────────────────────────────────────────

SCRIPTS = {
    'tgn': 'models/tgn/train.py',
}

if model not in SCRIPTS:
    raise ValueError(f"Unknown model '{model}'. Available: {list(SCRIPTS.keys())}")

print(f'Model:   {model}')
print(f'Dataset: {dataset}')
print(f'Script:  {SCRIPTS[model]}')

## 2. Install Dependencies

In [ ]:
DEPS = {
    'tgn': 'torch-geometric py-tgb numba wandb',
}

import subprocess
subprocess.run(f'pip install -q {DEPS[model]}', shell=True, check=True)
print('Dependencies installed.')

## 3. Train

In [ ]:
import sys

args = f'--dataset {dataset}'
if epochs   is not None: args += f' --epochs {epochs}'
if patience is not None: args += f' --patience {patience}'
if seed     is not None: args += f' --seed {seed}'
if device   is not None: args += f' --device {device}'

cmd = f'{sys.executable} {SCRIPTS[model]} {args}'
print(f'Running: {cmd}\n')
subprocess.run(cmd, shell=True, check=True)

## 4. Done

Checkpoint saved to `models/{model}/checkpoints/{dataset}/run{seed}.pkl`.

To evaluate, update `config.yaml`:
```yaml
model: tgn
checkpoints:
  - models/tgn/checkpoints/tgbl-wiki/run0.pkl
```
Then run `eval.ipynb`.